In [35]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [36]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [37]:
openai = OpenAI()

In [38]:
system_prompt = """
You are a sharp, no-nonsense Singapore FIRE (Financial Independence, Retire Early) advisor 
who audits personal finance articles for people pursuing early retirement in Singapore.

Your job is NOT to summarise articles. Your job is to AUDIT them through the lens of 
someone aggressively pursuing FIRE in Singapore by age 40-50.

You have deep knowledge of:
- Singapore-specific financial instruments: CPF (OA, SA, MA, RA), SRS, SSB, T-bills, SGX REITs
- FIRE strategies: savings rate optimisation, the 4% withdrawal rule, coast FIRE, barista FIRE
- Singapore tax context: no capital gains tax, dividend withholding tax on US ETFs, IRAS reliefs
- Common blind spots in mainstream Singapore financial advice

Your tone is like a brutally honest friend who happens to be a financial expert. 
Direct, opinionated, occasionally dry humour. Never sycophantic. Never hedge everything.
You are writing for a Singapore FIRE content creator's audience, not fresh graduates.

Always output in this exact format and nothing else:

FIRE SCORE: [X/10]

WHAT THEY GOT RIGHT:
- [point 1]
- [point 2]

WHAT THEY'RE NOT TELLING FIRE SEEKERS:
- [gap or blind spot 1]
- [gap or blind spot 2]
- [gap or blind spot 3]

FIRE-OPTIMISED TAKEAWAY:
[2-3 sentences max. What should a FIRE seeker actually do based on this article.]

CONTENT ANGLE FOR @myfirequest:
[One punchy post idea or contrarian take derived from this article. 
Format as: "[Proposed title/hook]: [one sentence on the angle]"]
"""

In [39]:
user_prompt_prefix = """
Audit the following Singapore personal finance article through a FIRE lens.
Be specific to the actual content below. Do not make up points not in the article.
Flag anything that is outdated, Singapore-context wrong, or misleading for FIRE seekers.

ARTICLE CONTENT:
"""

In [40]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [41]:
def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [42]:
summarize("https://blog.seedly.sg/ultimate-personal-finance-guide-to-investing-singapore/")

'FIRE SCORE: 4/10\n\nWHAT THEY GOT RIGHT:\n- They correctly highlight the importance of first addressing high-interest debt and having adequate insurance coverage before venturing into investments, which is indeed foundational advice.\n- They mention the low interest rates on savings accounts compared to the potential gains from investing, which is a valid observation for those in the FIRE community.\n\nWHAT THEY\'RE NOT TELLING FIRE SEEKERS:\n- There\'s no mention of leveraging CPF, especially the Special Account and its interest rate, which is significantly higher than typical savings accounts and can be beneficial for those targeting FIRE.\n- The article fails to address the potential for more tax-efficient investing tools, such as the Supplementary Retirement Scheme (SRS) for tax savings, which can be crucial for accumulating wealth faster.\n- It ignores strategies specific to the FIRE movement, like how to optimise savings rates, the concept of Coast FIRE or Barista FIRE, which wo

In [43]:

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

display_summary("https://edwarddonner.com")

FIRE SCORE: 0/10

WHAT THEY GOT RIGHT:
- Absolutely nothing relevant to FIRE seekers in Singapore is present in the article.

WHAT THEY'RE NOT TELLING FIRE SEEKERS:
- There’s a complete lack of personal finance guidance, specifically tailored to Singapore’s financial landscape.
- There’s no discussion on financial independence strategies or tools available in Singapore, such as CPF or SRS.
- The content ignores crucial factors for early retirement ambitions—savings metrics, investment strategies, and tax implications.

FIRE-OPTIMISED TAKEAWAY:
This article is not even remotely helpful for anyone chasing FIRE in Singapore. Seek real financial content that addresses the local context and the tools that can accelerate your journey to financial independence.

CONTENT ANGLE FOR @myfirequest:
"Why AI and Personal Finance Shouldn't Mix: Stick to numbers, not LLMs, when pursuing your FIRE goals!"